# Checkpoint and continue adapter training

Train a small low-rank output adapter, save all continuation state, reconstruct it, and compare resumed training with an uninterrupted run. You should understand a PyTorch optimizer and state_dict. The offline GRU and bundled text are a lifecycle fixture, not a pretrained LLM or a response-quality benchmark.

## Before you begin

Run the cells in order in a fresh kernel. Keep the complete repository and bundled tutorial data together. The setup below locates your checkout; in Colab it accepts a complete source ZIP. The notebook imports the adjacent Python implementation, so operators and resources have one canonical source.


In [ ]:
from pathlib import Path
import importlib.util
import io
import subprocess
import sys
import zipfile

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "jayrun").is_dir() and (p / "tutorials").is_dir()), None)
if ROOT is None and IN_COLAB:
    from google.colab import files
    uploaded = files.upload()  # Upload the complete Jayrun project ZIP, not just this notebook.
    if len(uploaded) != 1:
        raise ValueError("upload exactly one project ZIP")
    destination = Path("/content/jayrun_tutorial_source")
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
        if sum(item.file_size for item in archive.infolist()) > 500_000_000:
            raise ValueError("archive exceeds this notebook's extraction limit")
        for item in archive.infolist():
            path = (destination / item.filename).resolve()
            if not path.is_relative_to(destination.resolve()) or (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("unsafe archive path or symbolic link")
        archive.extractall(destination)
    candidates = [destination, *destination.iterdir()]
    ROOT = next((p for p in candidates if (p / "jayrun").is_dir() and (p / "tutorials").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Run Jupyter from the extracted project, or upload the full project ZIP in Colab.")
sys.path.insert(0, str(ROOT))
print("Tutorial source:", ROOT)


## Prepare the environment

The next cell checks the packages this lesson uses. Locally, install missing tutorial packages before continuing. In Colab, it installs only missing non-PyTorch packages. For PyTorch lessons, provide a suitable existing PyTorch build; the cell does not replace it.


In [ ]:
required = {'packaging': 'packaging>=25', 'numpy': 'numpy', 'torch': 'torch'}
missing = [requirement for name, requirement in required.items() if importlib.util.find_spec(name) is None]
# Colab provides PyTorch. Never replace an existing CPU/CUDA installation here.
if "torch" in missing:
    raise RuntimeError("Install the PyTorch build appropriate for your hardware before running this lesson.")
if missing:
    if not IN_COLAB:
        raise RuntimeError(f"Install these optional tutorial dependencies first: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
from packaging.version import Version
import packaging
if Version(packaging.__version__) < Version("25"):
    raise RuntimeError("This source archive requires packaging >= 25; update it explicitly before proceeding.")
import torch
torch.set_num_threads(1)  # Keep the small CPU demonstration bounded.


## 1. Separate shared input from mutable training state

The token dataset is shared read-only by convention. TrainingState owns model, optimizer, scheduler, sampler RNG, identity, update position, and examples-seen count. One graph iteration performs up to three updates, records progress, and reaches a control boundary. total_updates is an absolute target, not a number to add to the current position.


In [ ]:
import asyncio
import inspect
import tempfile
import torch
from pathlib import Path
from tutorials import adapter_finetuning as lesson

print(inspect.getsource(lesson.TrainingState))
print(inspect.getsource(lesson.TrainBlock))
dataset = lesson.load_fixture()
print("Train / validation shapes:", tuple(dataset.train.shape), tuple(dataset.validation.shape))


## 2. Train to update six

new_fixture_state prepares the small base and freezes it before scheduling. Only the low-rank head trains. run_training assembles and submits the confirmed graph using those canonical operators. Six updates correspond to 24 sampled examples in this recipe; optimizer and scheduler should both be populated.


In [ ]:
partial = await asyncio.to_thread(
    lesson.run_training, lesson.new_fixture_state(), dataset, total_updates=6)
print("Updates / examples seen:", partial.step, partial.examples_seen)
assert partial.step == 6 and partial.examples_seen == 24
assert partial.optimizer.state


## 3. Save, reconstruct, and continue

Saving model weights alone would lose optimizer momentum, schedule position, and the next sampled batch. The application checkpoint stores those values together and verifies dataset/model identity when loading. A new state is reconstructed, then restored before another context is submitted. Here the temporary directory is removed after restoration; keep a chosen output directory for a real experiment.


In [ ]:
with tempfile.TemporaryDirectory() as temporary:
    checkpoint = Path(temporary) / "training_checkpoint.pt"
    lesson.save_checkpoint(partial, dataset, checkpoint)
    restored = lesson.restore_checkpoint(lesson.new_fixture_state(), dataset, checkpoint)
assert restored.step == 6 and restored.examples_seen == 24
torch.testing.assert_close(restored.generator.get_state(), partial.generator.get_state(), rtol=0, atol=0)

resumed = await asyncio.to_thread(lesson.run_training, restored, dataset, total_updates=12)
print("After continuation:", resumed.step, resumed.examples_seen)
assert resumed.step == 12 and resumed.examples_seen == 48


## 4. Compare the full parameter state

Build the same initial fixture and train uninterrupted to update twelve. On this CPU recipe, resumed parameters should agree exactly. The canonical complete demo also compares direct PyTorch training and checks that frozen parameters did not change. These checks establish continuation for this fixture, not arbitrary stochastic models or hardware.


In [ ]:
uninterrupted = await asyncio.to_thread(
    lesson.run_training, lesson.new_fixture_state(), dataset, total_updates=12)
for name, value in uninterrupted.model.state_dict().items():
    torch.testing.assert_close(value, resumed.model.state_dict()[name], rtol=0, atol=0)
print("Checkpoint continuation matches uninterrupted CPU training.")

summary = await asyncio.to_thread(lesson.run_demo)
assert summary["matches_direct_baseline"] and summary["frozen_parameters_unchanged"]
print({key: summary[key] for key in ("updates", "trainable_parameters", "validation_loss_before", "validation_loss_after")})


## Try it yourself

Change the checkpoint boundary to update three and keep the final target twelve. Adapt the expected position/examples-seen checks and compare the result again. Try restoring into a fixture with a different seed: model identity should reject it.

Do not retry a partially executed optimizer block as if its effects were rolled back. Restore a known application checkpoint first. The sampler owns the RNG in this recipe; arbitrary dropout, worker RNG, or CUDA nondeterminism needs additional qualification. History records do not reconstruct training state.


## Optional: a pretrained transformer

The separate peft_finetuning implementation uses Transformers and PEFT. It downloads weights and needs explicit optional-package installation and a suitable memory budget. Enable only after completing the offline lesson. Record and reuse the resolved immutable revision. This tiny corpus does not evaluate production model quality.


In [ ]:
RUN_PRETRAINED = False
if RUN_PRETRAINED:
    from tutorials.peft_finetuning import run_pretrained
    pretrained_result = await asyncio.to_thread(
        run_pretrained, model_id="HuggingFaceTB/SmolLM2-135M", revision="main",
        output=Path("peft_results"), device="cpu", total_updates=12)
    print(pretrained_result)
